In [0]:
%pip install python-dotenv==1.0.1 -q

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys

repo_root = os.path.dirname(os.getcwd())  # notebook lives in <repo>/notebooks
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ["API_BASE_URL"] = "https://ev-project-navy-mu.vercel.app"
os.environ["API_USERNAME"] = dbutils.secrets.get("voltgrid", "api-username")
os.environ["API_PASSWORD"] = dbutils.secrets.get("voltgrid", "api-password")
os.environ["PII_SALT"] = dbutils.secrets.get("voltgrid", "pii-salt")
os.environ["DB_PASSWORD"] = "not-used-on-databricks"

from ingestion.http_client import login, make_session

session = make_session(login())
print("login OK")

In [0]:
import time

from config.settings import API_BASE_URL

url = f"{API_BASE_URL}/api/db/payments/"

for size in (100, 500, 1000):
    t = time.time()
    r = session.get(url, params={"page": 1, "page_size": size}, timeout=60)
    n = len(r.json()["data"]) if r.ok else r.text[:80]
    print(f"page_size {size} -> status {r.status_code}, rows {n}, {time.time() - t:.1f}s")

for since in ("2020-01-01T00:00:00Z", "2030-01-01T00:00:00Z"):
    r = session.get(url, params={"page": 1, "page_size": 2, "updated_after": since}, timeout=60)
    total = r.json()["pagination"]["total"] if r.ok else r.text[:80]
    print(f"updated_after {since[:10]} -> status {r.status_code}, total {total}")